### 3.5 Выбор спецификации модели

**Входы**

- Результаты EDA, попарного анализа и проверки гипотез
- Постановка задачи и критерии оценивания

**Выходы**

- Варианты спецификации: состав признаков, обработка, семейства моделей и настройки для сравнения

**Что делаем**

Определим варианты устройства будущей модели на основе накопленных выводов

In [ ]:
# dev-only: подготовка рабочей папки для интерактивного запуска
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


#### Какие варианты спецификации сравним

| Вариант | Основная идея |
| --- | --- |
| Dummy | Одинаковый прогноз для всех заявок — доля дефолтов в train |
| Логистическая регрессия | Складывает взвешенные вклады признаков и переводит результат в оценку риска |
| Случайный лес | Усредняет прогнозы множества деревьев решений |
| Градиентный бустинг на деревьях | Добавляет деревья постепенно: каждое уточняет общий результат |

Так задаём несколько вариантов устройства будущей модели: от простого базового прогноза до более гибких зависимостей. После обучения сравним полученные результаты по validation

Оставляем все 11 признаков. Связь двух столбцов ещё не означает, что один из них бесполезен для прогноза. Грейд и ставку сохраняем: по нашей постановке они известны до прогноза

При обучении используем train. Полученные варианты сравним по AP на validation, затем там же выберем порог по F2:

$$F_2=\frac{5TP}{5TP+FP+4FN}$$

При равной F2 берём больший порог. После этого решение не меняем и проверяем его на test. На train + validation заново не обучаем, чтобы не менять прогнозы, для которых уже выбрали порог

**Ограничение:** общий EDA и начальная очистка сделаны до разбиения. Поэтому test нельзя считать полностью невиданными данными. При этом обработка пропусков, обучение, выбор модели и порога не используют test. Случайное разбиение также не проверяет работу на будущем периоде или на данных другого банка